# Trellis displays: lattice and Cleveland

Trellis displays come from Becker, Cleveland and Shyu (*The visual design and control of
trellis display*, 1996), built into S-PLUS and later R's **lattice**. Their rules are
anytrellis' defaults: shared scales, axes only on the outer panels, labels in strips over
the panels, and panels ordered by what they show rather than alphabetically.

**Using a plot:** click it, then scroll to zoom along x (Shift+scroll: y), drag to pan, and
press `r` to reset. Panels that share a scale zoom and pan together. Hover a mark for its
values. Double-click a panel to see it alone (Page Up and Page Down: the next ones;
Backspace: all again), and press `f` (or the toolbar's button) for full screen.

In [ ]:
import polars as pl

from anytrellis import Trellis
from anytrellis.datasets import load

## Barley: a dot plot ordered by the median

The classic trellis display, from Cleveland's *Visualizing Data* (1993): barley yields of
ten varieties at six sites in Minnesota, in 1931 and 1932. In lattice:

```r
dotplot(variety ~ yield | site, data = barley, groups = year, layout = c(2, 3))
```

Sites and varieties are ordered by their median yield. Look at Morris: there, and only
there, 1932 beat 1931 for nearly every variety. Cleveland suggested the two years at Morris
had been swapped by mistake (later work found it may be real); either way, ordered panels
make it jump out.

In [ ]:
barley = load("barley").with_columns(pl.col("year").cast(pl.String))  # years as categories

dots = (
    Trellis(barley, wrap="site", cols=2, panel_height=160, title="Barley yields at six sites",
            x_title="yield (bushels per acre)", y_title="")
    .order("site", by="median", of="yield", descending=True)
    .order("variety", by="median", of="yield")
    .points(x="yield", y="variety", color="year", size=7)
)
dots

## Anscombe's quartet

Four datasets with the same means, variances, correlation and least-squares line (1973),
and four very different pictures. Each panel draws its points and the line fitted to
them: a second table with the `dataset` column lands in its own panels.

In [ ]:
quartet = load("anscombe")
fitted = (
    quartet.group_by("dataset")
    .agg(slope=pl.cov("x", "y") / pl.col("x").var(), mx=pl.col("x").mean(), my=pl.col("y").mean())
    .join(pl.DataFrame({"x": [3.0, 20.0]}), how="cross")
    .with_columns(y=pl.col("my") + pl.col("slope") * (pl.col("x") - pl.col("mx")))
)

anscombe = (
    Trellis(quartet, wrap="dataset", title="Anscombe's quartet")
    .points(x="x", y="y", size=8)
    .line(fitted, x="x", y="y", color="#d62728", name="least squares")
)
anscombe

## Conditioning on two variables

lattice's `xyplot(mpg ~ horsepower | cylinders * origin, groups = year)`: a grid
conditions on two columns. `share_x="col"` gives each column of the grid its own x scale.

In [ ]:
cars = load("cars").filter(pl.col("cylinders").is_in([4, 6, 8]))

conditioned = (
    Trellis(cars, row="origin", col="cylinders", panel_height=120,
            title="Fuel use and power, by cylinders and origin")
    .share(x="col")
    .points(x="horsepower", y="mpg", color="year", size=5, tooltip=["name"])
)
conditioned

## Small multiples instead of an animation

Hans Rosling's famous animation of fertility against life expectancy, as one panel per
year: every year is on screen at once, on the same scales.

In [ ]:
gapminder = load("gapminder")

years = (
    Trellis(gapminder, wrap="year", cols=4, panel_height=150,
            title="Fertility and life expectancy, 1955-2005")
    .points(x="fertility", y="life_expect", color="region", size=6, tooltip=["country", "pop"])
)
years